In [ ]:
!pip install -q biopython torch_geometric fair-esm

In [ ]:
#!/usr/bin/env python3
"""
T3 -- GNN-oracle reliability on MODELED structures (Kaggle, GPU).
Confirmed runnable environment (2026-07-18 Kaggle log): torch 2.10.0+cu128,
torch_geometric 2.8.0, biopython 1.87, fair-esm 2.0.0, GPU present.
Self-fetches candidates from RCSB if not pre-staged (see _fetch_candidates_via_rcsb).

Locked threshold, from ../PREREGISTRATION.md:
    PASS-for-the-proposal: AUROC on modeled structures >= 0.65 AND within
    0.10 AUROC of the crystal-structure number.
    FAIL: AUROC on modeled structures <= 0.60, OR a drop of >0.15 AUROC
    from crystal to modeled.

Pipeline (all implemented below, real -- not stubbed)
------------------------------------------------------
1. Download PDB coordinate files for candidate IDs (files.rcsb.org).
2. Parse chains with biopython. Identify the peptide chain as the shortest
   polymer chain (8-25 residues, standard MHC-I/II presented peptide length);
   the two next-shortest long chains among the remainder are treated as
   TCR alpha/beta (variable-domain chains are ~100-130 residues and, in a
   solved ternary complex, are the ones in contact with the peptide -- see
   below); everything else is MHC. This is a *structural* proxy for CDR3
   identification (contact-based), not full IMGT-anchored numbering --
   documented explicitly as a simplification, since IMGT anchoring requires
   a germline-alignment step this scaffold does not implement. Contact-based
   selection is defensible here because what the GNN actually needs is the
   interface, not the IMGT loop boundaries per se.
3. Interface graph: nodes = residues (any chain) with a CA within
   CROP_RADIUS_A of any peptide-chain CA; edges = residue pairs with CA-CA
   distance <= CROP_RADIUS_A; node features = one-hot amino acid (20) +
   one-hot role (TCR-contact-chain / MHC / peptide).
4. Labels: positive = the real solved complex's own interface graph.
   Negative (decoy): swap in a DIFFERENT candidate's peptide chain
   (structurally implausible re-pairing -- a real, if crude, negative
   construction; documented so decoy quality can be judged directly).
   One decoy generated per positive, matched at the complex level.
5. Train a 3-layer GCN (torch_geometric) for graph classification, held out
   at the COMPLEX level (not residue level).
6. "Modeled" structures: attempt ESMFold (fair-esm) on the TCR alpha/beta +
   peptide sequence of each held-out complex; if ESMFold is unavailable or
   fails (weight download, memory, time), fall back to an explicitly-labeled
   coordinate-perturbation proxy (Gaussian noise, sigma=1.5 Angstrom on all
   atoms, calibrated to typical AlphaFold/ESMFold-class backbone RMSD) --
   which structure source was actually used is recorded in the report, not
   silently substituted.
7. Evaluate the SAME trained GNN on (a) held-out crystal graphs and
   (b) held-out modeled/perturbed graphs. Report AUROC for both, apply the
   locked threshold via write_report().

Usage (Kaggle: New Notebook -> enable Internet + GPU -> paste/upload -> Run All)
"""
from __future__ import annotations
import json, os, random, time, urllib.request, io
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()
REPO = Path("/kaggle/working") if IS_KAGGLE else Path(__file__).resolve().parent.parent
PROC = REPO / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
STRUCT_DIR = REPO / "data" / "raw" / "pdb_structures"
STRUCT_DIR.mkdir(parents=True, exist_ok=True)

IN_CANDIDATES = PROC / "rcsb_tcr_pmhc_candidates.csv"
OUT_REPORT = PROC / "T3_gnn_oracle_report.json"

CROP_RADIUS_A = 5.0
BAR_MIN_MODELED_AUROC = 0.65
BAR_MAX_CRYSTAL_MODELED_DROP = 0.15
BAR_MAX_CRYSTAL_MODELED_WINDOW = 0.10

MAX_STRUCTURES = 220          # cap for wall-clock; report the cap explicitly
MIN_USABLE_COMPLEXES = 20     # below this, AUROC is too noisy to trust -- report UNDERPOWERED
TEST_FRACTION = 0.2
SEED = 13
AA20 = "ACDEFGHIKLMNPQRSTVWY"

random.seed(SEED)


def _fetch_candidates_via_rcsb(out_csv: Path, max_rows: int = 2000):
    import pandas as pd
    SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
    GRAPHQL_URL = "https://data.rcsb.org/graphql"

    def search_ids():
        query = {"query": {"type": "group", "logical_operator": "and", "nodes": [
            {"type": "terminal", "service": "full_text", "parameters": {"value": "T cell receptor"}},
            {"type": "terminal", "service": "full_text", "parameters": {"value": "major histocompatibility complex"}},
        ]}, "return_type": "entry",
            "request_options": {"return_all_hits": False, "paginate": {"start": 0, "rows": max_rows}}}
        req = urllib.request.Request(SEARCH_URL, data=json.dumps(query).encode(),
                                      headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=30) as r:
            res = json.load(r)
        return [x["identifier"] for x in res.get("result_set", [])], res.get("total_count", 0)

    def fetch_metadata_batch(ids_batch):
        gql = """query($ids: [String!]!) { entries(entry_ids: $ids) { rcsb_id struct { title }
                 rcsb_entry_info { deposited_polymer_entity_instance_count resolution_combined } } }"""
        body = json.dumps({"query": gql, "variables": {"ids": ids_batch}}).encode()
        req = urllib.request.Request(GRAPHQL_URL, data=body, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=30) as r:
            res = json.load(r)
        rows = []
        for e in res.get("data", {}).get("entries", []) or []:
            if e is None:
                continue
            info = e.get("rcsb_entry_info") or {}
            res_list = info.get("resolution_combined") or []
            rows.append({"pdb_id": e["rcsb_id"], "title": (e.get("struct") or {}).get("title"),
                         "n_chains": info.get("deposited_polymer_entity_instance_count"),
                         "resolution": res_list[0] if res_list else None})
        return rows

    ids, total = search_ids()
    print(f"[self-fetch] {len(ids)} candidate IDs fetched (RCSB reports {total} total full-text hits)")
    rows = []
    for i in range(0, len(ids), 50):
        batch = ids[i:i + 50]
        try:
            rows.extend(fetch_metadata_batch(batch))
        except Exception as e:
            print(f"[warn] batch {i} failed: {e}")
        time.sleep(0.2)
    df = pd.DataFrame(rows)
    df_filtered = df[df["n_chains"].fillna(0) >= 4].copy()
    df_filtered.to_csv(out_csv, index=False)
    return df_filtered


def download_structure(pdb_id: str) -> Path | None:
    out = STRUCT_DIR / f"{pdb_id}.pdb"
    if out.exists():
        return out
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            data = r.read()
        out.write_bytes(data)
        return out
    except Exception as e:
        print(f"[warn] download failed {pdb_id}: {e}")
        return None


def parse_complex(pdb_path: Path):
    """Return dict(chain_id -> list[(resname, CA-coord)]) for standard-AA
    residues only, or None if parsing fails / too few chains."""
    from Bio.PDB import PDBParser
    from Bio.PDB.Polypeptide import three_to_index, index_to_one
    import numpy as np
    parser = PDBParser(QUIET=True)
    try:
        struct = parser.get_structure(pdb_path.stem, str(pdb_path))
    except Exception:
        return None
    model = next(struct.get_models())
    chains = {}
    for chain in model:
        residues = []
        for res in chain:
            if "CA" not in res:
                continue
            try:
                idx = three_to_index(res.get_resname())
                aa = index_to_one(idx)
            except Exception:
                continue
            residues.append((aa, np.array(res["CA"].coord, dtype=float)))
        if len(residues) >= 5:
            chains[chain.id] = residues
    if len(chains) < 3:
        return None
    return chains


def assign_roles(chains: dict):
    """Peptide = shortest chain in [7,25] residues. TCR contact chains =
    the two chains (excluding peptide) with a CA within CROP_RADIUS_A of the
    peptide chain, ranked by contact-residue count. MHC = remaining chains
    that also contact the peptide. Documented structural (not IMGT) proxy."""
    import numpy as np
    lengths = {cid: len(res) for cid, res in chains.items()}
    pep_candidates = [cid for cid, n in lengths.items() if 7 <= n <= 25]
    if not pep_candidates:
        return None
    pep_id = min(pep_candidates, key=lambda c: lengths[c])
    pep_coords = np.stack([c for _, c in chains[pep_id]])

    contact_counts = {}
    for cid, res in chains.items():
        if cid == pep_id:
            continue
        coords = np.stack([c for _, c in res])
        d = np.linalg.norm(coords[:, None, :] - pep_coords[None, :, :], axis=-1)
        n_contact = int((d.min(axis=1) <= CROP_RADIUS_A).sum())
        if n_contact > 0:
            contact_counts[cid] = n_contact
    if len(contact_counts) < 2:
        return None
    ranked = sorted(contact_counts, key=lambda c: -contact_counts[c])
    tcr_chains = ranked[:2]
    mhc_chains = ranked[2:]
    return {"peptide": pep_id, "tcr": tcr_chains, "mhc": mhc_chains}


def build_interface_graph(chains: dict, roles: dict, jitter_sigma: float = 0.0):
    """nodes = residues within CROP_RADIUS_A of the peptide chain (any role);
    edges = CA-CA distance <= CROP_RADIUS_A; features = one-hot AA (20) +
    one-hot role (tcr/mhc/peptide, 3). jitter_sigma>0 adds Gaussian noise to
    coordinates BEFORE graph construction -- the documented modeled-structure
    proxy used when ESMFold is unavailable."""
    import numpy as np
    pep_id = roles["peptide"]
    role_of = {pep_id: "peptide"}
    for c in roles["tcr"]:
        role_of[c] = "tcr"
    for c in roles["mhc"]:
        role_of[c] = "mhc"

    all_res = []  # (aa, coord, role)
    for cid, res_list in chains.items():
        if cid not in role_of:
            continue
        for aa, coord in res_list:
            c = coord + (np.random.normal(0, jitter_sigma, size=3) if jitter_sigma > 0 else 0.0)
            all_res.append((aa, c, role_of[cid]))
    if len(all_res) < 6:
        return None
    coords = np.stack([r[1] for r in all_res])
    pep_mask = np.array([r[2] == "peptide" for r in all_res])
    if pep_mask.sum() == 0:
        return None
    pep_coords = coords[pep_mask]
    d_to_pep = np.linalg.norm(coords[:, None, :] - pep_coords[None, :, :], axis=-1).min(axis=1)
    keep = d_to_pep <= CROP_RADIUS_A
    if keep.sum() < 6:
        return None
    kept = [all_res[i] for i in range(len(all_res)) if keep[i]]
    kcoords = coords[keep]

    role_idx = {"tcr": 0, "mhc": 1, "peptide": 2}
    feats = []
    for aa, _, role in kept:
        onehot_aa = [1.0 if aa == a else 0.0 for a in AA20]
        onehot_role = [0.0, 0.0, 0.0]
        onehot_role[role_idx[role]] = 1.0
        feats.append(onehot_aa + onehot_role)

    n = len(kept)
    dmat = np.linalg.norm(kcoords[:, None, :] - kcoords[None, :, :], axis=-1)
    edges = np.argwhere((dmat <= CROP_RADIUS_A) & (dmat > 0))
    return {"x": feats, "edge_index": edges.T.tolist(), "n_nodes": n}


def make_decoy_graph(pos_chains: dict, pos_roles: dict, other_chains: dict, other_roles: dict):
    """Decoy: keep this complex's TCR+MHC, replace the peptide chain's
    coordinates with a DIFFERENT complex's peptide (structurally implausible
    re-pairing). Recentered on the original peptide's centroid so the swap
    lands near the real binding site rather than off in space."""
    import numpy as np
    pep_id = pos_roles["peptide"]
    orig_pep = pos_chains[pep_id]
    orig_centroid = np.mean([c for _, c in orig_pep], axis=0)

    other_pep_id = other_roles["peptide"]
    other_pep = other_chains[other_pep_id]
    other_centroid = np.mean([c for _, c in other_pep], axis=0)
    shift = orig_centroid - other_centroid
    swapped_pep = [(aa, c + shift) for aa, c in other_pep]

    merged = dict(pos_chains)
    merged[pep_id] = swapped_pep
    return build_interface_graph(merged, pos_roles)


def graphs_to_pyg(records):
    import torch
    from torch_geometric.data import Data
    out = []
    for rec in records:
        x = torch.tensor(rec["x"], dtype=torch.float)
        ei = rec["edge_index"]
        edge_index = torch.tensor(ei, dtype=torch.long) if len(ei) and len(ei[0]) else torch.zeros((2, 0), dtype=torch.long)
        y = torch.tensor([rec["y"]], dtype=torch.long)
        out.append(Data(x=x, edge_index=edge_index, y=y))
    return out


class GCN(__import__("torch").nn.Module):
    def __init__(self, in_dim=23, hidden=32):
        import torch.nn as nn
        from torch_geometric.nn import GCNConv, global_mean_pool
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.lin = nn.Linear(hidden, 2)
        self.pool = global_mean_pool

    def forward(self, x, edge_index, batch):
        import torch.nn.functional as F
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = self.pool(x, batch)
        return self.lin(x)


def train_gnn(train_graphs, epochs=40):
    import torch
    from torch_geometric.loader import DataLoader
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = GCN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loader = DataLoader(train_graphs, batch_size=16, shuffle=True)
    model.train()
    for ep in range(epochs):
        tot = 0.0
        for batch in loader:
            batch = batch.to(device)
            opt.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = torch.nn.functional.cross_entropy(out, batch.y)
            loss.backward()
            opt.step()
            tot += float(loss) * batch.num_graphs
        if ep % 10 == 0:
            print(f"  [train] epoch {ep} loss {tot / max(1, len(train_graphs)):.4f}")
    return model, device


def eval_auroc(model, device, graphs):
    import torch
    from torch_geometric.loader import DataLoader
    from sklearn.metrics import roc_auc_score
    model.eval()
    ys, scores = [], []
    with torch.no_grad():
        for batch in DataLoader(graphs, batch_size=32):
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            prob1 = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            scores.extend(prob1.tolist())
            ys.extend(batch.y.cpu().numpy().tolist())
    if len(set(ys)) < 2:
        return None, len(ys)
    return float(roc_auc_score(ys, scores)), len(ys)


def try_esmfold_structure(chains: dict, roles: dict):
    """Attempt real ESMFold structure prediction on the TCR-contact chains +
    peptide sequence (fair-esm). Returns a modeled `chains` dict on success,
    or None on any failure (weights unavailable, OOM, timeout) -- caller
    falls back to coordinate-perturbation. This is attempted once per
    process (model load is expensive) and reused."""
    global _ESMFOLD_MODEL
    try:
        import torch
        if "_ESMFOLD_MODEL" not in globals() or _ESMFOLD_MODEL is None:
            import esm
            print("  [esmfold] loading esmfold_v1 (first call only, may take minutes)...")
            _ESMFOLD_MODEL = esm.pretrained.esmfold_v1()
            _ESMFOLD_MODEL = _ESMFOLD_MODEL.eval()
            if torch.cuda.is_available():
                _ESMFOLD_MODEL = _ESMFOLD_MODEL.cuda()
        model = _ESMFOLD_MODEL
        modeled = {}
        for role_key, cids in (("tcr", roles["tcr"]), ("mhc", roles["mhc"]), ("peptide", [roles["peptide"]])):
            for cid in cids:
                seq = "".join(aa for aa, _ in chains[cid])
                if len(seq) < 3 or len(seq) > 400:
                    return None
                with torch.no_grad():
                    out = model.infer_pdb(seq)
                import io as _io
                from Bio.PDB import PDBParser
                p = PDBParser(QUIET=True)
                s = p.get_structure(cid, _io.StringIO(out))
                m = next(s.get_models())
                ch = next(m.get_chains())
                res_list = []
                from Bio.PDB.Polypeptide import three_to_index, index_to_one
                import numpy as np
                for res in ch:
                    if "CA" not in res:
                        continue
                    try:
                        idx = three_to_index(res.get_resname())
                        aa2 = index_to_one(idx)
                    except Exception:
                        continue
                    res_list.append((aa2, np.array(res["CA"].coord, dtype=float)))
                if len(res_list) < 3:
                    return None
                modeled[cid] = res_list
        return modeled
    except Exception as e:
        print(f"  [esmfold] unavailable/failed ({type(e).__name__}: {e}) -- will use perturbation proxy")
        return None


def main():
    import pandas as pd
    import numpy as np

    if not IN_CANDIDATES.exists():
        print(f"[info] {IN_CANDIDATES} not found -- self-fetching from RCSB.")
        cands = _fetch_candidates_via_rcsb(IN_CANDIDATES)
    else:
        cands = pd.read_csv(IN_CANDIDATES)
    print(f"[load] {len(cands)} candidate PDB IDs")

    ids = cands["pdb_id"].dropna().unique().tolist()
    random.shuffle(ids)
    ids = ids[:MAX_STRUCTURES]

    usable = []  # list of dict(pdb_id, chains, roles)
    for i, pdb_id in enumerate(ids):
        path = download_structure(pdb_id)
        if path is None:
            continue
        chains = parse_complex(path)
        if chains is None:
            continue
        roles = assign_roles(chains)
        if roles is None:
            continue
        usable.append({"pdb_id": pdb_id, "chains": chains, "roles": roles})
        if i % 25 == 0:
            print(f"  [parse] {i}/{len(ids)} attempted, {len(usable)} usable so far")

    n_usable = len(usable)
    print(f"[parse] {n_usable}/{len(ids)} candidates yielded a usable TCR:peptide:MHC interface")

    if n_usable < MIN_USABLE_COMPLEXES:
        report = {
            "status": "UNDERPOWERED",
            "n_candidates_attempted": len(ids),
            "n_usable_complexes": n_usable,
            "min_required": MIN_USABLE_COMPLEXES,
            "note": "Too few candidates parsed into a usable TCR:peptide:MHC interface "
                    "(download failures, chain-role heuristic rejects, etc.) to train/eval "
                    "meaningfully. Report this, do not force an AUROC.",
        }
        OUT_REPORT.write_text(json.dumps(report, indent=2))
        print(f"[done] UNDERPOWERED, wrote {OUT_REPORT}")
        return

    random.shuffle(usable)
    n_test = max(1, int(n_usable * TEST_FRACTION))
    test_set = usable[:n_test]
    train_set = usable[n_test:]
    print(f"[split] {len(train_set)} train complexes / {len(test_set)} test complexes (complex-level)")

    def positives_and_decoys(items, pool):
        recs = []
        for it in items:
            pos_g = build_interface_graph(it["chains"], it["roles"])
            if pos_g is None:
                continue
            pos_g["y"] = 1
            recs.append(pos_g)
            other = random.choice([o for o in pool if o["pdb_id"] != it["pdb_id"]])
            dec_g = make_decoy_graph(it["chains"], it["roles"], other["chains"], other["roles"])
            if dec_g is None:
                continue
            dec_g["y"] = 0
            recs.append(dec_g)
        return recs

    train_records = positives_and_decoys(train_set, usable)
    test_records_crystal = positives_and_decoys(test_set, usable)
    print(f"[graphs] {len(train_records)} train graphs, {len(test_records_crystal)} test (crystal) graphs")

    if len(train_records) < 6 or len(test_records_crystal) < 4:
        report = {"status": "UNDERPOWERED_AFTER_GRAPH_CONSTRUCTION",
                   "n_train_graphs": len(train_records), "n_test_graphs": len(test_records_crystal)}
        OUT_REPORT.write_text(json.dumps(report, indent=2))
        print(f"[done] UNDERPOWERED, wrote {OUT_REPORT}")
        return

    train_pyg = graphs_to_pyg(train_records)
    test_pyg_crystal = graphs_to_pyg(test_records_crystal)

    model, device = train_gnn(train_pyg)
    auroc_crystal, n_crystal = eval_auroc(model, device, test_pyg_crystal)
    print(f"[eval] crystal AUROC = {auroc_crystal} (n={n_crystal})")

    # Modeled-structure eval: try ESMFold per held-out complex, else perturbation proxy.
    modeled_records = []
    esmfold_used = 0
    perturb_used = 0
    for it in test_set:
        modeled_chains = try_esmfold_structure(it["chains"], it["roles"])
        if modeled_chains is not None:
            merged = dict(it["chains"])
            merged.update(modeled_chains)
            g = build_interface_graph(merged, it["roles"])
            esmfold_used += 1
        else:
            g = build_interface_graph(it["chains"], it["roles"], jitter_sigma=1.5)
            perturb_used += 1
        if g is not None:
            g["y"] = 1
            modeled_records.append(g)
            other = random.choice([o for o in usable if o["pdb_id"] != it["pdb_id"]])
            dg = make_decoy_graph(it["chains"], it["roles"], other["chains"], other["roles"])
            if dg is not None:
                dg["y"] = 0
                modeled_records.append(dg)

    modeled_structure_source = ("esmfold" if perturb_used == 0 else
                                 "perturbation_proxy" if esmfold_used == 0 else "mixed")
    print(f"[modeled] source={modeled_structure_source} (esmfold={esmfold_used}, perturbation={perturb_used})")

    test_pyg_modeled = graphs_to_pyg(modeled_records)
    auroc_modeled, n_modeled = eval_auroc(model, device, test_pyg_modeled)
    print(f"[eval] modeled AUROC = {auroc_modeled} (n={n_modeled}, source={modeled_structure_source})")

    if auroc_crystal is None or auroc_modeled is None:
        report = {"status": "DEGENERATE_TEST_SET",
                   "note": "Test set ended up single-class after graph construction failures; "
                           "AUROC undefined. Report raw counts, do not force a number.",
                   "n_crystal": n_crystal if auroc_crystal is not None else 0,
                   "n_modeled": n_modeled if auroc_modeled is not None else 0}
        OUT_REPORT.write_text(json.dumps(report, indent=2))
        print(f"[done] wrote {OUT_REPORT}: DEGENERATE_TEST_SET")
        return

    write_report(auroc_crystal, auroc_modeled, len(test_set),
                 decoy_strategy="peptide-chain swap from a different candidate complex, "
                                 "recentered on original peptide centroid",
                 modeled_structure_source=modeled_structure_source,
                 n_usable_complexes=n_usable, n_candidates_attempted=len(ids))


def write_report(auroc_crystal, auroc_modeled, n_test_complexes, decoy_strategy,
                  modeled_structure_source="unknown", n_usable_complexes=None, n_candidates_attempted=None):
    drop = auroc_crystal - auroc_modeled
    within_window = abs(drop) <= BAR_MAX_CRYSTAL_MODELED_WINDOW
    passes = (auroc_modeled >= BAR_MIN_MODELED_AUROC) and within_window
    fails = (auroc_modeled <= 0.60) or (drop > BAR_MAX_CRYSTAL_MODELED_DROP)
    verdict = "PASS" if passes else ("FAIL" if fails else "AMBIGUOUS -- report raw numbers, don't force a call")
    report = {
        "n_candidates_attempted": n_candidates_attempted,
        "n_usable_complexes": n_usable_complexes,
        "n_test_complexes": n_test_complexes,
        "decoy_strategy": decoy_strategy,
        "modeled_structure_source": modeled_structure_source,
        "auroc_crystal_structures": auroc_crystal,
        "auroc_modeled_structures": auroc_modeled,
        "crystal_to_modeled_drop": drop,
        "bars": {"min_modeled_auroc": BAR_MIN_MODELED_AUROC,
                 "max_crystal_modeled_window": BAR_MAX_CRYSTAL_MODELED_WINDOW,
                 "fail_drop_threshold": BAR_MAX_CRYSTAL_MODELED_DROP},
        "verdict": verdict,
    }
    with open(OUT_REPORT, "w") as f:
        json.dump(report, f, indent=2)
    print(f"[done] wrote {OUT_REPORT}: {verdict}")


if __name__ == "__main__":
    main()